# CloudSimLLM Validation (§6.1)

Compare simulator output against real vLLM measurements. Produces:
1. Per-metric error table (mean / p99 absolute and relative error)
2. Q-Q plots (TTFT, TPOT, throughput)
3. Scatter plots colored by (input_len, output_len, concurrency)
4. PDF figures saved to `figures/` ready for paper inclusion

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGDIR = Path('figures'); FIGDIR.mkdir(exist_ok=True)

# Paper-quality matplotlib defaults
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'legend.fontsize': 9, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [ ]:
# ----- Inputs (set the paths to your measurement / simulation files) -----
REAL_PATH = 'measurements_a100_llama3_8b.json'      # from run_vllm_benchmark.py
SIM_PATH  = 'sim_a100_llama3_8b.csv'                # exported from LlmStatistics

real = pd.DataFrame(json.load(open(REAL_PATH)))
sim  = pd.read_csv(SIM_PATH)  # columns: input_len, output_len, concurrency, ttft_p50, ttft_p99, tpot_p50, tpot_p99, throughput_tok_per_s

key = ['input_len', 'output_len', 'concurrency']
df = real.merge(sim, on=key, suffixes=('_real', '_sim'))
print(f'Matched {len(df)} cells')
df.head()

## 1. Error table

In [ ]:
metrics = ['ttft_p50', 'ttft_p99', 'tpot_p50', 'tpot_p99', 'throughput_tok_per_s']
rows = []
for m in metrics:
    real_v = df[f'{m}_real'].values
    sim_v  = df[f'{m}_sim'].values
    abs_err = np.abs(sim_v - real_v)
    rel_err = abs_err / np.maximum(1e-9, real_v)
    rows.append({
        'metric': m,
        'mean_abs_err': abs_err.mean(),
        'p99_abs_err':  np.quantile(abs_err, 0.99),
        'mean_rel_err_%': 100 * rel_err.mean(),
        'p99_rel_err_%':  100 * np.quantile(rel_err, 0.99),
    })
errtab = pd.DataFrame(rows).set_index('metric')
errtab.style.format('{:.3f}')

In [ ]:
errtab.to_csv(FIGDIR / 'table_validation_errors.csv')
with open(FIGDIR / 'table_validation_errors.tex', 'w') as f:
    f.write(errtab.to_latex(float_format='%.3f'))
print('wrote', FIGDIR / 'table_validation_errors.tex')

## 2. Q-Q plots — distribution-level agreement

In [ ]:
def qq(ax, real_v, sim_v, title, unit):
    qs = np.linspace(0.01, 0.99, 99)
    qr = np.quantile(real_v, qs); qs_ = np.quantile(sim_v, qs)
    lo = min(qr.min(), qs_.min()); hi = max(qr.max(), qs_.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.6)
    ax.plot(qr, qs_, 'o', ms=3.5, alpha=0.8)
    ax.set_xlabel(f'real ({unit})'); ax.set_ylabel(f'simulated ({unit})')
    ax.set_title(title); ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
qq(axes[0], df.ttft_p50_real, df.ttft_p50_sim, 'TTFT (median per cell)', 's')
qq(axes[1], df.tpot_p50_real * 1000, df.tpot_p50_sim * 1000, 'TPOT (median per cell)', 'ms')
qq(axes[2], df.throughput_tok_per_s_real, df.throughput_tok_per_s_sim, 'Throughput', 'tok/s')
fig.tight_layout()
fig.savefig(FIGDIR / 'fig_qq_validation.pdf')
fig.savefig(FIGDIR / 'fig_qq_validation.png')

## 3. Residual heatmap — where does the simulator drift?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, m in zip(axes, ['ttft_p50', 'tpot_p50']):
    pivot = df.assign(rel=100*(df[f'{m}_sim'] - df[f'{m}_real'])/df[f'{m}_real'])\
              .pivot_table(index='input_len', columns='concurrency', values='rel', aggfunc='mean')
    im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-15, vmax=15, aspect='auto')
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
    ax.set_xlabel('concurrency'); ax.set_ylabel('input tokens')
    ax.set_title(f'{m}: relative error (%)')
    plt.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout(); fig.savefig(FIGDIR / 'fig_residual_heatmap.pdf')

## 4. Pass/Fail check — paper claim is mean ≤ 8%

In [ ]:
TARGET = {'ttft_p50': 8, 'ttft_p99': 8, 'tpot_p50': 6, 'tpot_p99': 6, 'throughput_tok_per_s': 5}
for m, t in TARGET.items():
    actual = errtab.loc[m, 'mean_rel_err_%']
    flag = '\u2705' if actual <= t else '\u274c'
    print(f'{flag}  {m:30s}  mean_rel_err = {actual:5.2f}%   target ≤ {t}%')